# Итоговое обучение MDEN

- Все LOG_AGE 2 s и 30 s
- Единый временной шаг 30 s
- Split по аккумуляторам
- Streaming training

In [ ]:
import json
import math
import re
from dataclasses import fields
from itertools import chain
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from mden_battery.data import (
    PreparedBatchIterableDataset,
    WindowConfig,
)
from mden_battery.loss import MDENJointLoss
from mden_battery.model import MDEN, MDENConfig
from mden_battery.training import seed_everything

In [ ]:
PROJECT_ROOT = Path(
    "/home/jupyter/datasphere/Klim_PiNN/PiNN_v2_Klim"
)
RAW_DATA_DIR = Path(
    "/home/jupyter/datasphere/Klim_PiNN/DataSet/"
    "battery_aging_log_data_v2/extracted/"
    "10.35097-kww7jv8ajuvchcah/data/dataset/"
    "cell_log_age"
)
CONFIG_PATH = PROJECT_ROOT / "configs" / "article_mden.yaml"
PREPARED_ROOT = (
    PROJECT_ROOT / "data" / "prepared_log_age_30s_10_cells"
)
PARTS_DIR = PREPARED_ROOT / "parts"
MANIFEST_PATH = PREPARED_ROOT / "manifest.csv"
INDEX_PATH = PREPARED_ROOT / "prepared_index.csv"
SPLITS_PATH = PREPARED_ROOT / "cell_splits.csv"
SCALER_PATH = PREPARED_ROOT / "scaler.csv"
PREPROCESSING_PATH = PREPARED_ROOT / "preprocessing.json"

OUTPUT_DIR = PROJECT_ROOT / "runs" / "mden_final_10_cells"
BEST_PATH = OUTPUT_DIR / "best.pt"
LAST_PATH = OUTPUT_DIR / "last.pt"

CSV_SEPARATOR = ";"
RATED_CAPACITY_AH = 3.0
TARGET_INTERVAL_S = 30
CHUNK_SIZE = 500_000
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15
SPLIT_SEED = 42
REBUILD_PREPARED_DATA = False
PREPROCESSING_VERSION = 3
DATASET_CELL_LIMIT = 10
DATASET_SELECTION_SEED = 42
MICRO_BATCH_SIZE = 2048
GRADIENT_ACCUMULATION_STEPS = 1
TRAIN_NUM_WORKERS = 2
EVAL_NUM_WORKERS = 1
PREFETCH_FACTOR = 4
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4

INPUT_COLUMNS = [
    "cycle",
    "time_s",
    "voltage_V",
    "current_A",
    "temperature_C",
]
REQUIRED_RAW_COLUMNS = [
    "timestamp_s",
    "EFC",
    "v_raw_V",
    "i_raw_A",
    "t_cell_degC",
    "soc_est",
    "cap_aged_est_Ah",
]
LOG_FILE_PATTERN = re.compile(
    r"cell_log_age_(?P<interval>2|30)s_"
    r"P(?P<parameter>\d+)_(?P<replicate>\d+)_"
    r"S(?P<slave>\d+)_C(?P<channel>\d+)\.csv$",
    re.IGNORECASE,
)
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
USE_AMP = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = USE_AMP

In [ ]:
def discover_log_files(root: Path) -> pd.DataFrame:
    """Находит LOG_AGE-файлы и извлекает идентификаторы."""
    records = []
    for path in root.rglob("cell_log_age_*s_*.csv"):
        match = LOG_FILE_PATTERN.fullmatch(path.name)
        if match is None:
            continue
        values = match.groupdict()
        cell_id = (
            f"P{int(values['parameter']):03d}_"
            f"{int(values['replicate'])}_"
            f"S{int(values['slave']):02d}_"
            f"C{int(values['channel']):02d}"
        )
        records.append(
            {
                "cell_id": cell_id,
                "sampling_interval_s": int(values["interval"]),
                "path": str(path.resolve()),
            }
        )
    if not records:
        raise FileNotFoundError(
            f"LOG_AGE CSV files were not found under {root}"
        )
    return (
        pd.DataFrame(records)
        .sort_values(["cell_id", "sampling_interval_s", "path"])
        .reset_index(drop=True)
    )

In [ ]:
def select_cells(
    manifest: pd.DataFrame,
    cell_limit: int,
    seed: int,
) -> pd.DataFrame:
    """Выбирает фиксированное подмножество аккумуляторов."""
    cell_ids = manifest["cell_id"].drop_duplicates().to_numpy()
    if cell_limit < 3:
        raise ValueError("Для train/val/test нужно минимум 3 аккумулятора")
    if len(cell_ids) < cell_limit:
        raise ValueError(
            f"Найдено {len(cell_ids)} аккумуляторов, нужно {cell_limit}"
        )

    rng = np.random.default_rng(seed)
    rng.shuffle(cell_ids)
    selected = set(cell_ids[:cell_limit])
    return (
        manifest[manifest["cell_id"].isin(selected)]
        .sort_values(["cell_id", "sampling_interval_s", "path"])
        .reset_index(drop=True)
    )

In [ ]:
def split_cells(
    cell_ids: list[str],
    seed: int,
) -> dict[str, str]:
    """Разделяет аккумуляторы без пересечения между выборками."""
    cells = np.asarray(sorted(set(cell_ids)), dtype=object)
    if len(cells) < 3:
        raise ValueError("Для train/val/test нужно минимум три cell_id")

    rng = np.random.default_rng(seed)
    rng.shuffle(cells)
    test_count = max(1, round(len(cells) * TEST_FRACTION))
    val_count = max(1, round(len(cells) * VAL_FRACTION))
    if test_count + val_count >= len(cells):
        raise ValueError("Недостаточно аккумуляторов для split")

    split_map = {str(cell): "train" for cell in cells}
    for cell in cells[:test_count]:
        split_map[str(cell)] = "test"
    for cell in cells[test_count : test_count + val_count]:
        split_map[str(cell)] = "val"
    return split_map

In [ ]:
def load_capacity_anchors(
    path: Path,
) -> tuple[np.ndarray, np.ndarray]:
    """Загружает точки ёмкости для интерполяции SOH по EFC."""
    anchor_parts = []
    iterator = pd.read_csv(
        path,
        sep=CSV_SEPARATOR,
        usecols=["EFC", "cap_aged_est_Ah"],
        chunksize=CHUNK_SIZE,
        low_memory=False,
    )
    for chunk in iterator:
        anchors = pd.DataFrame(
            {
                "efc": pd.to_numeric(
                    chunk["EFC"],
                    errors="coerce",
                ),
                "capacity": pd.to_numeric(
                    chunk["cap_aged_est_Ah"],
                    errors="coerce",
                ),
            }
        )
        anchors = anchors.replace(
            [np.inf, -np.inf], np.nan
        ).dropna()
        anchors = anchors[anchors["capacity"] > 0]
        if not anchors.empty:
            anchor_parts.append(anchors)

    if not anchor_parts:
        return np.array([]), np.array([])

    anchors = pd.concat(anchor_parts, ignore_index=True)
    anchors = (
        anchors.groupby("efc", as_index=False)["capacity"]
        .median()
        .sort_values("efc")
    )
    return (
        anchors["efc"].to_numpy(np.float64),
        anchors["capacity"].to_numpy(np.float64),
    )

In [ ]:
def downsample_chunk(
    chunk: pd.DataFrame,
    source_interval_s: int,
    row_offset: int,
) -> pd.DataFrame:
    """Приводит 2-секундные данные к единому шагу 30 секунд."""
    if TARGET_INTERVAL_S % source_interval_s != 0:
        raise ValueError(
            f"Unsupported interval: {source_interval_s} s"
        )
    factor = TARGET_INTERVAL_S // source_interval_s
    positions = np.arange(
        row_offset,
        row_offset + len(chunk),
    )
    selected = positions % factor == 0
    return chunk.iloc[np.flatnonzero(selected)].reset_index(drop=True)

In [ ]:
def transform_chunk(
    chunk: pd.DataFrame,
    anchor_efc: np.ndarray,
    anchor_capacity: np.ndarray,
) -> pd.DataFrame:
    """Преобразует raw chunk и восстанавливает SOC/SOH labels."""
    numeric = chunk.apply(pd.to_numeric, errors="coerce")
    efc = numeric["EFC"].to_numpy(np.float64)
    capacity = np.interp(
        efc,
        anchor_efc,
        anchor_capacity,
        left=anchor_capacity[0],
        right=anchor_capacity[-1],
    )
    soc = numeric["soc_est"] / 100.0
    soc = soc.where(soc.between(-0.01, 1.01))

    frame = pd.DataFrame(
        {
            "cycle": numeric["EFC"],
            "time_s": numeric["timestamp_s"],
            "voltage_V": numeric["v_raw_V"],
            "current_A": numeric["i_raw_A"],
            "temperature_C": numeric["t_cell_degC"],
            "soc": soc,
            "soh": capacity / RATED_CAPACITY_AH,
        }
    )
    return (
        frame.replace([np.inf, -np.inf], np.nan)
        .dropna()
        .reset_index(drop=True)
    )

In [ ]:
def write_prepared_frame(
    frame: pd.DataFrame,
    path_without_suffix: Path,
) -> Path:
    """Сохраняет prepared chunk в Parquet или CSV.GZ."""
    parquet_path = path_without_suffix.with_suffix(".parquet")
    try:
        frame.to_parquet(parquet_path, index=False)
        return parquet_path
    except (ImportError, ValueError):
        csv_path = path_without_suffix.with_suffix(".csv.gz")
        frame.to_csv(
            csv_path,
            index=False,
            compression="gzip",
        )
        return csv_path

In [ ]:
def update_statistics(
    count: int,
    mean: np.ndarray,
    m2: np.ndarray,
    values: np.ndarray,
) -> tuple[int, np.ndarray, np.ndarray]:
    """Обновляет устойчивые streaming-статистики признаков."""
    batch_count = len(values)
    if batch_count == 0:
        return count, mean, m2

    batch_mean = values.mean(axis=0)
    batch_m2 = np.square(values - batch_mean).sum(axis=0)
    total_count = count + batch_count
    delta = batch_mean - mean
    total_mean = mean + delta * batch_count / total_count
    total_m2 = (
        m2
        + batch_m2
        + np.square(delta) * count * batch_count / total_count
    )
    return total_count, total_mean, total_m2

In [ ]:
def prepare_source_file(
    row: pd.Series,
    split: str,
    statistics: tuple[int, np.ndarray, np.ndarray],
) -> tuple[
    list[dict],
    tuple[int, np.ndarray, np.ndarray],
    str,
]:
    """Подготавливает один LOG_AGE-файл ограниченными chunks."""
    path = Path(row["path"])
    anchor_efc, anchor_capacity = load_capacity_anchors(path)
    if anchor_efc.size == 0:
        return [], statistics, "missing_capacity"

    index_rows = []
    row_offset = 0
    iterator = pd.read_csv(
        path,
        sep=CSV_SEPARATOR,
        usecols=REQUIRED_RAW_COLUMNS,
        chunksize=CHUNK_SIZE,
        low_memory=False,
    )
    for chunk_index, raw_chunk in enumerate(iterator):
        raw_rows = len(raw_chunk)
        chunk = downsample_chunk(
            raw_chunk,
            int(row["sampling_interval_s"]),
            row_offset,
        )
        row_offset += raw_rows
        frame = transform_chunk(
            chunk,
            anchor_efc,
            anchor_capacity,
        )
        if frame.empty:
            continue

        part_name = (
            f"{split}_{row['cell_id']}_"
            f"{int(row['sampling_interval_s'])}s_"
            f"part{chunk_index:05d}"
        )
        part_path = write_prepared_frame(
            frame,
            PARTS_DIR / part_name,
        )
        index_rows.append(
            {
                "cell_id": row["cell_id"],
                "split": split,
                "source": str(path.resolve()),
                "path": str(part_path.resolve()),
                "rows": len(frame),
                "chunk": chunk_index,
                "sampling_interval_s": TARGET_INTERVAL_S,
            }
        )
        if split == "train":
            values = frame[INPUT_COLUMNS].to_numpy(np.float64)
            statistics = update_statistics(*statistics, values)

    status = "prepared" if index_rows else "empty"
    return index_rows, statistics, status

In [ ]:
def preprocessing_signature(manifest: pd.DataFrame) -> dict:
    """Возвращает параметры текущей версии preprocessing."""
    return {
        "version": PREPROCESSING_VERSION,
        "raw_data_dir": str(RAW_DATA_DIR),
        "file_count": len(manifest),
        "cell_ids": sorted(manifest["cell_id"].unique().tolist()),
        "dataset_cell_limit": DATASET_CELL_LIMIT,
        "dataset_selection_seed": DATASET_SELECTION_SEED,
        "target_interval_s": TARGET_INTERVAL_S,
        "rated_capacity_ah": RATED_CAPACITY_AH,
        "soh_formula": "cap_aged_est_Ah / rated_capacity_ah",
        "soh_interpolation": "linear_over_efc",
        "boundary_policy": "nearest_anchor",
        "split_seed": SPLIT_SEED,
        "train_fraction": TRAIN_FRACTION,
        "val_fraction": VAL_FRACTION,
        "test_fraction": TEST_FRACTION,
    }

In [ ]:
def prepare_dataset(manifest: pd.DataFrame) -> None:
    """Готовит все файлы и train-only scaler в streaming-режиме."""
    PREPARED_ROOT.mkdir(parents=True, exist_ok=True)
    PARTS_DIR.mkdir(parents=True, exist_ok=True)
    split_map = split_cells(
        manifest["cell_id"].astype(str).tolist(),
        SPLIT_SEED,
    )
    manifest = manifest.copy()
    manifest["split"] = manifest["cell_id"].map(split_map)

    feature_count = len(INPUT_COLUMNS)
    statistics = (
        0,
        np.zeros(feature_count, dtype=np.float64),
        np.zeros(feature_count, dtype=np.float64),
    )
    index_rows = []
    statuses = []
    progress = tqdm(
        manifest.iterrows(),
        total=len(manifest),
        desc="Preprocessing",
        unit="file",
        dynamic_ncols=True,
    )
    for row_index, row in progress:
        split = str(row["split"])
        rows, statistics, status = prepare_source_file(
            row,
            split,
            statistics,
        )
        index_rows.extend(rows)
        statuses.append(status)
        progress.set_postfix(
            split=split,
            status=status,
        )

    manifest["status"] = statuses
    index = pd.DataFrame(index_rows)
    count, mean, m2 = statistics
    if index.empty or count == 0:
        raise ValueError("Prepared train dataset is empty")
    std = np.sqrt(m2 / count)
    std = np.where(std == 0, 1.0, std)

    scaler = pd.DataFrame(
        {
            "feature": INPUT_COLUMNS,
            "mean": mean,
            "std": std,
            "count": count,
        }
    )
    splits = pd.DataFrame(
        [
            {"cell_id": cell_id, "split": split}
            for cell_id, split in sorted(split_map.items())
        ]
    )
    manifest.to_csv(MANIFEST_PATH, index=False)
    index.to_csv(INDEX_PATH, index=False)
    splits.to_csv(SPLITS_PATH, index=False)
    scaler.to_csv(SCALER_PATH, index=False)
    PREPROCESSING_PATH.write_text(
        json.dumps(
            preprocessing_signature(manifest),
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

In [ ]:
def prepared_dataset_is_current(manifest: pd.DataFrame) -> bool:
    """Проверяет наличие и версию prepared dataset."""
    required_paths = [
        INDEX_PATH,
        SPLITS_PATH,
        SCALER_PATH,
        PREPROCESSING_PATH,
    ]
    if not all(path.is_file() for path in required_paths):
        return False
    saved_signature = json.loads(
        PREPROCESSING_PATH.read_text(encoding="utf-8")
    )
    if saved_signature != preprocessing_signature(manifest):
        return False
    index = pd.read_csv(INDEX_PATH, usecols=["path"])
    return all(Path(path).is_file() for path in index["path"])

In [ ]:
def validate_prepared_dataset() -> pd.DataFrame:
    """Проверяет индекс, файлы частей и отсутствие split leakage."""
    index = pd.read_csv(INDEX_PATH)
    required_columns = {
        "cell_id",
        "split",
        "source",
        "path",
        "rows",
        "chunk",
    }
    missing_columns = required_columns - set(index.columns)
    if missing_columns:
        raise KeyError(
            f"Index misses columns: {sorted(missing_columns)}"
        )
    missing_paths = [
        path for path in index["path"]
        if not Path(path).is_file()
    ]
    if missing_paths:
        raise FileNotFoundError(missing_paths[:5])
    split_count = index.groupby("cell_id")["split"].nunique()
    if not split_count.eq(1).all():
        raise ValueError("Один cell_id найден в разных splits")
    if set(index["split"]) != {"train", "val", "test"}:
        raise ValueError("Требуются train, val и test splits")
    return index

In [ ]:
def load_config(
    path: Path,
) -> tuple[dict, MDENConfig]:
    """Загружает YAML и поддерживаемые параметры модели."""
    raw_config = yaml.safe_load(path.read_text(encoding="utf-8"))
    model_fields = {field.name for field in fields(MDENConfig)}
    model_config = MDENConfig(
        **{
            key: value
            for key, value in raw_config["model"].items()
            if key in model_fields
        }
    )
    return raw_config, model_config

In [ ]:
def build_loaders(config: dict) -> dict[str, DataLoader]:
    """Создаёт RAM-cached DataLoaders для всех splits."""
    data_config = config["data"]
    window_config = WindowConfig(
        input_length=int(data_config["input_length"]),
        horizon=int(data_config["horizon"]),
        stride=int(data_config["stride"]),
    )
    loaders = {}
    for split in ("train", "val", "test"):
        num_workers = (
            TRAIN_NUM_WORKERS
            if split == "train"
            else EVAL_NUM_WORKERS
        )
        dataset = PreparedBatchIterableDataset(
            INDEX_PATH,
            batch_size=MICRO_BATCH_SIZE,
            split=split,
            input_cols=data_config["input_columns"],
            config=window_config,
            scaler_csv=SCALER_PATH,
            shuffle_batches=split == "train",
            seed=int(config["training"]["seed"]),
        )
        loaders[split] = DataLoader(
            dataset,
            batch_size=None,
            num_workers=num_workers,
            pin_memory=DEVICE.type == "cuda",
            persistent_workers=num_workers > 0,
            prefetch_factor=PREFETCH_FACTOR,
        )
    return loaders

In [ ]:
def count_batches(
    index: pd.DataFrame,
    config: dict,
) -> dict[str, int]:
    """Вычисляет число batches для корректного tqdm total."""
    data_config = config["data"]
    batch_size = MICRO_BATCH_SIZE
    total_length = (
        int(data_config["input_length"])
        + int(data_config["horizon"])
    )
    stride = int(data_config["stride"])
    result = {}
    for split in ("train", "val", "test"):
        source_rows = (
            index[index["split"] == split]
            .groupby("source")["rows"]
            .sum()
        )
        source_windows = [
            max(0, 1 + (int(rows) - total_length) // stride)
            for rows in source_rows
        ]
        result[split] = sum(
            math.ceil(windows / batch_size)
            for windows in source_windows
            if windows > 0
        )
    return result

In [ ]:
def train_epoch_amp(
    model: torch.nn.Module,
    criterion: torch.nn.Module,
    loader,
    optimizer: torch.optim.Optimizer,
    grad_scaler: torch.cuda.amp.GradScaler,
    device: torch.device,
    total_batches: int,
    accumulation_steps: int,
    gradient_clip_norm: float,
) -> dict[str, float]:
    """Обучает эпоху с AMP и накоплением градиента."""
    model.train()
    criterion.train()
    optimizer.zero_grad(set_to_none=True)
    sums = {"loss": 0.0, "mse_soc": 0.0, "mse_soh": 0.0}
    samples = 0

    for batch_index, batch in enumerate(loader):
        x = batch["x"].to(device, non_blocking=True)
        y_soc = batch["y_soc"].to(device, non_blocking=True)
        y_soh = batch["y_soh"].to(device, non_blocking=True)
        group_start = batch_index // accumulation_steps
        group_start *= accumulation_steps
        group_size = min(
            accumulation_steps,
            total_batches - group_start,
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            losses = criterion(model(x), y_soc, y_soh)
            backward_loss = losses["loss"] / group_size

        grad_scaler.scale(backward_loss).backward()
        should_step = (
            (batch_index + 1) % accumulation_steps == 0
            or batch_index + 1 == total_batches
        )
        if should_step:
            grad_scaler.unscale_(optimizer)
            parameters = [
                parameter
                for group in optimizer.param_groups
                for parameter in group["params"]
                if parameter.grad is not None
            ]
            torch.nn.utils.clip_grad_norm_(
                parameters,
                gradient_clip_norm,
            )
            grad_scaler.step(optimizer)
            grad_scaler.update()
            optimizer.zero_grad(set_to_none=True)

        batch_size = x.shape[0]
        samples += batch_size
        for name in sums:
            sums[name] += float(losses[name].detach()) * batch_size

    if samples == 0:
        raise ValueError("Training loader produced no batches")
    return {name: value / samples for name, value in sums.items()}

In [ ]:
@torch.no_grad()
def eval_epoch_amp(
    model: torch.nn.Module,
    criterion: torch.nn.Module,
    loader,
    device: torch.device,
) -> dict[str, float]:
    """Вычисляет метрики эпохи с mixed precision."""
    model.eval()
    criterion.eval()
    sums = {"loss": 0.0, "mse_soc": 0.0, "mse_soh": 0.0}
    samples = 0

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        y_soc = batch["y_soc"].to(device, non_blocking=True)
        y_soh = batch["y_soh"].to(device, non_blocking=True)
        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=USE_AMP,
        ):
            losses = criterion(model(x), y_soc, y_soh)

        batch_size = x.shape[0]
        samples += batch_size
        for name in sums:
            sums[name] += float(losses[name].detach()) * batch_size

    if samples == 0:
        raise ValueError("Evaluation loader produced no batches")
    return {name: value / samples for name, value in sums.items()}

In [ ]:
def save_checkpoint(
    path: Path,
    model: MDEN,
    criterion: MDENJointLoss,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.ReduceLROnPlateau,
    grad_scaler: torch.cuda.amp.GradScaler,
    config: dict,
    scaler: pd.DataFrame,
    epoch: int,
    val_loss: float,
) -> None:
    """Сохраняет обучение и параметры preprocessing."""
    torch.save(
        {
            "model": model.state_dict(),
            "criterion": criterion.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "grad_scaler": grad_scaler.state_dict(),
            "config": config,
            "micro_batch_size": MICRO_BATCH_SIZE,
            "gradient_accumulation_steps": (
                GRADIENT_ACCUMULATION_STEPS
            ),
            "scaler": scaler.to_dict(orient="list"),
            "preprocessing": json.loads(
                PREPROCESSING_PATH.read_text(encoding="utf-8")
            ),
            "epoch": epoch,
            "val_loss": val_loss,
        },
        path,
    )

In [ ]:
def plot_history(metrics: dict[str, list[float]]) -> None:
    """Строит графики loss и validation RMSE."""
    epochs = np.arange(1, len(metrics["train_loss"]) + 1)
    _, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(epochs, metrics["train_loss"], label="train")
    axes[0].plot(epochs, metrics["val_loss"], label="validation")
    axes[0].set_title("Joint loss")

    axes[1].plot(
        epochs,
        np.sqrt(metrics["val_soc_mse"]),
        label="SOC RMSE",
    )
    axes[1].plot(
        epochs,
        np.sqrt(metrics["val_soh_mse"]),
        label="SOH RMSE",
    )
    axes[1].set_title("Validation RMSE")

    for axis in axes:
        axis.set_xlabel("Эпоха")
        axis.grid(alpha=0.3)
        axis.legend()
    plt.tight_layout()
    plt.show()

## Подготовка данных

In [ ]:
assert (PROJECT_ROOT / "pyproject.toml").is_file()
assert RAW_DATA_DIR.is_dir(), RAW_DATA_DIR
assert CONFIG_PATH.is_file(), CONFIG_PATH

full_manifest = discover_log_files(RAW_DATA_DIR)
manifest = select_cells(
    full_manifest,
    DATASET_CELL_LIMIT,
    DATASET_SELECTION_SEED,
)
interval_summary = (
    manifest.groupby("sampling_interval_s")
    .size()
    .rename("files")
    .reset_index()
)
print(interval_summary.to_string(index=False))
print("selected files:", len(manifest))
print("selected cells:", manifest["cell_id"].nunique())
print("cell_ids:", sorted(manifest["cell_id"].unique()))

is_current = prepared_dataset_is_current(manifest)
if REBUILD_PREPARED_DATA or not is_current:
    prepare_dataset(manifest)

index = validate_prepared_dataset()
split_summary = (
    index.groupby("split", as_index=False)
    .agg(
        cells=("cell_id", "nunique"),
        files=("source", "nunique"),
        rows=("rows", "sum"),
    )
    .sort_values("split")
)
print(split_summary.to_string(index=False))

## Инициализация

In [ ]:
raw_config, model_config = load_config(CONFIG_PATH)
train_config = raw_config["training"]
seed_everything(int(train_config["seed"]))

assert model_config.input_dim == len(
    raw_config["data"]["input_columns"]
)
assert model_config.horizon == int(raw_config["data"]["horizon"])

loaders = build_loaders(raw_config)
batch_counts = count_batches(index, raw_config)
if any(count == 0 for count in batch_counts.values()):
    raise ValueError(f"Empty split: {batch_counts}")

model = MDEN(model_config).to(DEVICE)
criterion = MDENJointLoss(
    feature_dim=model_config.input_dim
).to(DEVICE)
optimizer = torch.optim.AdamW(
    chain(model.parameters(), criterion.parameters()),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fused=DEVICE.type == "cuda",
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)
grad_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler = pd.read_csv(SCALER_PATH)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

history = {
    "train_loss": [],
    "val_loss": [],
    "train_soc_mse": [],
    "val_soc_mse": [],
    "train_soh_mse": [],
    "val_soh_mse": [],
}
best_val_loss = float("inf")
epochs_without_improvement = 0

print("device:", DEVICE)
print("batches:", batch_counts)
print("mixed precision:", USE_AMP)
print("micro batch:", MICRO_BATCH_SIZE)
print(
    "effective batch:",
    MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
)

## Обучение

- Первый запуск кеширует выборки в RAM.

In [ ]:
epoch_progress = tqdm(
    range(1, int(train_config["epochs"]) + 1),
    desc="Training",
    unit="epoch",
    dynamic_ncols=True,
    mininterval=1.0,
)
for epoch in epoch_progress:
    train_loader = tqdm(
        loaders["train"],
        total=batch_counts["train"],
        desc=f"Train {epoch}",
        unit="batch",
        leave=False,
        dynamic_ncols=True,
        mininterval=1.0,
    )
    train_metrics = train_epoch_amp(
        model,
        criterion,
        train_loader,
        optimizer,
        grad_scaler,
        DEVICE,
        batch_counts["train"],
        GRADIENT_ACCUMULATION_STEPS,
        float(
            train_config["gradient_clip_norm"]
        ),
    )

    val_loader = tqdm(
        loaders["val"],
        total=batch_counts["val"],
        desc=f"Validation {epoch}",
        unit="batch",
        leave=False,
        dynamic_ncols=True,
        mininterval=1.0,
    )
    val_metrics = eval_epoch_amp(
        model,
        criterion,
        val_loader,
        DEVICE,
    )
    scheduler.step(val_metrics["loss"])

    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])
    history["train_soc_mse"].append(train_metrics["mse_soc"])
    history["val_soc_mse"].append(val_metrics["mse_soc"])
    history["train_soh_mse"].append(train_metrics["mse_soh"])
    history["val_soh_mse"].append(val_metrics["mse_soh"])

    save_checkpoint(
        LAST_PATH,
        model,
        criterion,
        optimizer,
        scheduler,
        grad_scaler,
        raw_config,
        scaler,
        epoch,
        val_metrics["loss"],
    )
    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        epochs_without_improvement = 0
        save_checkpoint(
            BEST_PATH,
            model,
            criterion,
            optimizer,
            scheduler,
            grad_scaler,
            raw_config,
            scaler,
            epoch,
            best_val_loss,
        )
    else:
        epochs_without_improvement += 1

    epoch_progress.set_postfix(
        train=f"{train_metrics['loss']:.5f}",
        val=f"{val_metrics['loss']:.5f}",
    )
    if epochs_without_improvement >= int(
        train_config["patience"]
    ):
        break

## Test

In [ ]:
checkpoint = torch.load(
    BEST_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(checkpoint["model"])
criterion.load_state_dict(checkpoint["criterion"])
test_loader = tqdm(
    loaders["test"],
    total=batch_counts["test"],
    desc="Test",
    unit="batch",
    dynamic_ncols=True,
)
test_metrics = eval_epoch_amp(
    model,
    criterion,
    test_loader,
    DEVICE,
)
print(test_metrics)

## Кривые обучения

In [ ]:
plot_history(history)